In [1]:
"""
ESG Data Ingestion Pipeline
Sources: Wikirate API + CDP (CSV-based)
Target: Large EU-listed companies benchmarking

Requirements:
    pip install requests pandas tqdm python-dotenv

Usage:
    python esg_pipeline.py

Output:
    esg_combined.csv  — merged, normalised dataset ready for ML
"""

import os
import time
import requests
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

WIKIRATE_API_KEY = os.getenv("WIKIRATE_API_KEY", "")  # Optional — raises limit from 100 to 1000/hr
CDP_CSV_PATH = os.getenv("CDP_CSV_PATH", "cdp_data.csv")  # Path to your downloaded CDP CSV

OUTPUT_PATH = "esg_combined.csv"

# Large EU-listed companies to benchmark (ISIN or Wikirate company names)
# Extend this list as needed
TARGET_COMPANIES = [
    "LVMH", "TotalEnergies", "Volkswagen", "Siemens", "SAP",
    "Airbus", "BNP Paribas", "Allianz", "BASF", "Schneider Electric",
    "Unilever", "Philips", "Banco Santander", "ING Group", "Intesa Sanpaolo",
    "Enel", "ENI", "Iberdrola", "ASML", "Stellantis",
]

# ESG metrics to pull from Wikirate
# Full metric list: https://wikirate.org/metrics
WIKIRATE_METRICS = {
    "GHG_Scope1":        "GHG Emissions Scope 1",
    "GHG_Scope2":        "GHG Emissions Scope 2",
    "GHG_Scope3":        "GHG Emissions Scope 3",
    "Energy_Consumption": "Total Energy Consumption",
    "Women_Board":       "Percentage of Women on Board",
    "Employee_Count":    "Number of Employees",
    "Lost_Days":         "Lost Days due to Injury",
    "CEO_Pay_Ratio":     "CEO Pay Ratio",
    "Ethics_Policy":     "Code of Ethics",
    "Sustainability_Report": "Sustainability Report Published",
}

YEARS = [2021, 2022, 2023]

# ─────────────────────────────────────────────
# WIKIRATE INGESTION
# ─────────────────────────────────────────────

WIKIRATE_BASE = "https://wikirate.org"

def wikirate_headers():
    headers = {"Accept": "application/json"}
    if WIKIRATE_API_KEY:
        headers["X-API-Key"] = WIKIRATE_API_KEY
    return headers


def search_company_id(company_name: str) -> str | None:
    """Look up a Wikirate company ID by name."""
    url = f"{WIKIRATE_BASE}/Company.json"
    params = {"name": company_name, "limit": 1}
    try:
        r = requests.get(url, headers=wikirate_headers(), params=params, timeout=10)
        r.raise_for_status()
        items = r.json().get("items", [])
        if items:
            return items[0].get("name")  # Wikirate uses name as the key
    except Exception as e:
        print(f"  [WARN] Could not find '{company_name}': {e}")
    return None


def fetch_metric(company_name: str, metric_label: str, year: int) -> float | str | None:
    """Fetch a single metric value for a company and year."""
    # Encode metric name for URL
    metric_slug = metric_label.replace(" ", "_")
    url = f"{WIKIRATE_BASE}/{company_name}+{metric_slug}+{year}.json"
    try:
        r = requests.get(url, headers=wikirate_headers(), timeout=10)
        if r.status_code == 404:
            return None
        r.raise_for_status()
        data = r.json()
        return data.get("value")
    except Exception:
        return None


def ingest_wikirate() -> pd.DataFrame:
    """Pull all configured metrics for all target companies across all years."""
    print("\n📡 Ingesting from Wikirate API...")
    rows = []

    for company in tqdm(TARGET_COMPANIES, desc="Companies"):
        for year in YEARS:
            row = {"company": company, "year": year, "source": "wikirate"}
            for col_name, metric_label in WIKIRATE_METRICS.items():
                value = fetch_metric(company, metric_label, year)
                row[col_name] = value
                time.sleep(0.15)  # Respect rate limits (~6 req/s)
            rows.append(row)

    df = pd.DataFrame(rows)
    print(f"  ✓ Wikirate: {len(df)} rows, {len(df.columns)} columns")
    return df


# ─────────────────────────────────────────────
# CDP INGESTION
# ─────────────────────────────────────────────

# CDP column mapping — adjust to match your downloaded CSV headers
# Download from: https://www.cdp.net/en/responses (request research access)
CDP_COLUMN_MAP = {
    "Account Name":                          "company",
    "Year":                                  "year",
    "C6.1_Scope 1":                          "GHG_Scope1",
    "C6.3_Scope 2 (market-based)":           "GHG_Scope2",
    "C6.5_Scope 3 total":                    "GHG_Scope3",
    "C8.2_Total energy consumption":         "Energy_Consumption",
    "C1.2_Climate score":                    "CDP_Climate_Score",
    "W1.1_Water withdrawal total":           "Water_Withdrawal",
    "F1.1_Forest score":                     "CDP_Forest_Score",
}


def ingest_cdp() -> pd.DataFrame | None:
    """Load and normalise CDP CSV export."""
    path = Path(CDP_CSV_PATH)
    if not path.exists():
        print(f"\n⚠️  CDP CSV not found at '{CDP_CSV_PATH}'")
        print("   → Apply for research access at https://www.cdp.net/en/responses")
        print("   → Set CDP_CSV_PATH in your .env file once downloaded\n")
        return None

    print(f"\n📂 Loading CDP data from {CDP_CSV_PATH}...")
    df_raw = pd.read_csv(path, low_memory=False)

    # Keep only columns we care about
    available = {k: v for k, v in CDP_COLUMN_MAP.items() if k in df_raw.columns}
    df = df_raw[list(available.keys())].rename(columns=available)

    # Filter to target companies
    df = df[df["company"].isin(TARGET_COMPANIES)]

    # Filter to target years
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df = df[df["year"].isin(YEARS)]

    df["source"] = "cdp"

    print(f"  ✓ CDP: {len(df)} rows for {df['company'].nunique()} companies")
    return df


# ─────────────────────────────────────────────
# MERGE & NORMALISE
# ─────────────────────────────────────────────

def merge_sources(wikirate_df: pd.DataFrame, cdp_df: pd.DataFrame | None) -> pd.DataFrame:
    """Outer-join Wikirate and CDP on (company, year), deduplicate overlapping columns."""
    if cdp_df is None:
        return wikirate_df

    # Merge on company + year, prefer Wikirate for overlapping numeric cols
    merged = pd.merge(
        wikirate_df,
        cdp_df.drop(columns=["source"]),
        on=["company", "year"],
        how="outer",
        suffixes=("_wr", "_cdp"),
    )

    # Coalesce duplicate columns: prefer Wikirate, fall back to CDP
    for col in ["GHG_Scope1", "GHG_Scope2", "GHG_Scope3", "Energy_Consumption"]:
        col_wr = f"{col}_wr"
        col_cdp = f"{col}_cdp"
        if col_wr in merged.columns and col_cdp in merged.columns:
            merged[col] = merged[col_wr].combine_first(merged[col_cdp])
            merged.drop(columns=[col_wr, col_cdp], inplace=True)

    merged["source"] = "wikirate+cdp"
    return merged


def normalise(df: pd.DataFrame) -> pd.DataFrame:
    """Basic cleaning: types, missing value flags, sort."""
    numeric_cols = [
        "GHG_Scope1", "GHG_Scope2", "GHG_Scope3", "Energy_Consumption",
        "Women_Board", "Employee_Count", "CEO_Pay_Ratio",
        "Water_Withdrawal",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Coverage flag — % of numeric fields populated per row
    df["data_coverage_pct"] = df[
        [c for c in numeric_cols if c in df.columns]
    ].notna().mean(axis=1).round(2) * 100

    df.sort_values(["company", "year"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    print("=" * 55)
    print("  ESG Data Ingestion Pipeline")
    print("  Sources: Wikirate + CDP")
    print("=" * 55)

    wikirate_df = ingest_wikirate()
    cdp_df = ingest_cdp()

    combined = merge_sources(wikirate_df, cdp_df)
    combined = normalise(combined)

    combined.to_csv(OUTPUT_PATH, index=False)
    print(f"\n✅ Saved to {OUTPUT_PATH}")
    print(f"   Rows: {len(combined)} | Companies: {combined['company'].nunique()} | Years: {sorted(combined['year'].unique())}")
    print(f"   Avg data coverage: {combined['data_coverage_pct'].mean():.1f}%")

    # Quick preview
    print("\n── Sample Output ──────────────────────────────────")
    print(combined[["company", "year", "GHG_Scope1", "GHG_Scope2", "Women_Board", "data_coverage_pct"]].head(10).to_string(index=False))


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'dotenv'